In [1]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [2]:
import pickle, re, numpy as np
from pathlib import Path
import pdfplumber
from sentence_transformers import SentenceTransformer
import faiss

file_path = str(PROJECT_ROOT / "data/raw/pdf/2026_hope_ladder_selected.pdf")
INDEX_DIR = Path(file_path).parent / "rag_index"
INDEX_DIR.mkdir(exist_ok=True)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K = 5

print("PDF 존재:", Path(file_path).exists())
print("인덱스 저장 경로:", INDEX_DIR)

c:\Users\user\catcher-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF 존재: True
인덱스 저장 경로: c:\Users\user\catcher-llm\data\raw\pdf\rag_index


In [3]:
# ── 텍스트 추출 ──────────────────────────────────────────────
DEPT_PATTERN = re.compile(
    r"\s*[가-힣]*(성평등|보건복지|고용노동|국토교통|교육부|문화체육|행정안전|중소벤처|농림축산|환경부|법무부)[가-힣]*(부|처|청)?\s*"
)
SECTION_KEYS = ["지원대상", "핵심내용", "이용방법", "문의처"]

def extract_policy_name(lines):
    for line in lines[:6]:
        line = line.strip()
        if not line or len(line) < 4:
            continue
        kr = sum(1 for c in line if "가" <= c <= "힣")
        if kr / max(len(line), 1) < 0.4:
            continue
        name = DEPT_PATTERN.sub(" ", line).strip()
        if 4 <= len(name) <= 40:
            return name
    return lines[0].strip() if lines else "정책명 미상"

def extract_section(text, key):
    pattern = re.compile(rf"{key}[：:]?\s*(.*?)(?={'|'.join(SECTION_KEYS)}|$)", re.S)
    m = pattern.search(text)
    return m.group(1).strip()[:500] if m else ""

chunks = []
print("PDF 파싱 중...")
with pdfplumber.open(file_path) as pdf:
    for i, page in enumerate(pdf.pages):
        raw = page.extract_text() or ""
        if not raw.strip():
            continue
        lines = [l for l in raw.splitlines() if l.strip()]
        policy = extract_policy_name(lines)
        chunk = {
            "page": i + 1,
            "policy_name": policy,
            "target":  extract_section(raw, "지원대상"),
            "content": extract_section(raw, "핵심내용"),
            "method":  extract_section(raw, "이용방법"),
            "contact": extract_section(raw, "문의처"),
            "full_text": raw[:1000],
        }
        chunks.append(chunk)

print(f"총 {len(chunks)}개 청크 추출 완료")

# ── 임베딩 ────────────────────────────────────────────────────
print("임베딩 모델 로드 중...")
model = SentenceTransformer(EMBED_MODEL)

texts = []
for c in chunks:
    t = f"{c['policy_name']} {c['target']} {c['content']} {c['method']}"
    texts.append(t.strip())

print("임베딩 생성 중...")
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

# ── FAISS 인덱스 저장 ─────────────────────────────────────────
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, str(INDEX_DIR / "faiss.index"))
with open(INDEX_DIR / "chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print(f"인덱스 저장 완료 → {INDEX_DIR}")

PDF 파싱 중...
총 49개 청크 추출 완료
임베딩 모델 로드 중...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9387.56it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


임베딩 생성 중...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.89it/s]

인덱스 저장 완료 → c:\Users\user\catcher-llm\data\raw\pdf\rag_index


In [4]:
index = faiss.read_index(str(INDEX_DIR / "faiss.index"))
with open(INDEX_DIR / "chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
model = SentenceTransformer(EMBED_MODEL)
print(f"로드 완료 — 청크 {len(chunks)}개, 인덱스 {index.ntotal}개")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10806.14it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


로드 완료 — 청크 49개, 인덱스 49개


In [5]:
def search(query, model, index, chunks, top_k=TOP_K):
    qvec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qvec, top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), 1):
        c = chunks[idx]
        results.append({"rank": rank, "score": float(score), **c})
    return results

query = "임산부 지원 정책"          # ← 원하는 질문으로 변경
results = search(query, model, index, chunks)

print(f"\n🔍 '{query}' 검색 결과\n{'='*50}")
for r in results:
    print(f"\n[{r['rank']}위] {r['policy_name']}  (p.{r['page']}, 유사도 {r['score']:.3f})")
    if r["target"]:  print(f"  ▸ 지원대상: {r['target'][:80]}")
    if r["content"]: print(f"  ▸ 핵심내용: {r['content'][:80]}")
    if r["method"]:  print(f"  ▸ 이용방법: {r['method'][:80]}")


🔍 '임산부 지원 정책' 검색 결과

[1위] 모두의 정책 K-희망사다리 2026 121  (p.21, 유사도 0.703)
  ▸ 지원대상: •신 청일 기준 임산부
  ▸ 핵심내용: •임 신 후 받을 수 있는 각종 임신지원 서비스를 한 번에 안내받고 통합 신청하는
서비스
•전국 공통 서비스
구분 서비스
엽산제 지원, 철분제 
  ▸ 이용방법: •온라인 신청: 정부24(plus.gov.kr)
•방문 신청: 임산부 주소지 읍·면·동 행정복지센터 또는 보건소

[2위] 모두의 정책 K-희망사다리 2026 043  (p.2, 유사도 0.684)
  ▸ 지원대상: •모 든 20~49세 남녀 중 가임력 검사 희망자
- 결혼 여부 및 자녀 수 무관
- 15~19세 남녀 중 부부(예비부부, 사실혼 포함) 지원 
  ▸ 핵심내용: •사업 참여 의료기관*을 통한 필수 가임력 검사** 후 비용 지원
- 여성: 최대 13만 원, 남성: 최대 5만 원
*e보건소 → 민원서비스 →
  ▸ 이용방법: •온 라인 신청: e보건소 공공보건포털(e-health.go.kr)
•방 문 신청: 주소지 관할 보건소
•먼 저 신청 및 검사의뢰서 발급 후에 

[3위] 생애주기별 국민생활 서비스 - 가족·여성 122  (p.22, 유사도 0.653)
  ▸ 지원대상: • 신청자: 출산자(산모) 본인 또는 배우자
•대리인: 출산자(산모)의 친부모 또는 시부모
※ 대 리인은 방문 신청만 가능
  ▸ 핵심내용: •출 생신고와 함께 첫만남이용권, 부모급여 등 각종 출산 지원 서비스를 한 번에
안내받고 통합 신청하는 서비스
※ 출 생신고를 하지 않는 경우 

[4위] 생애주기별 국민생활 서비스 - 가족·여성 124  (p.23, 유사도 0.576)
  ▸ 지원대상: • 임신·출산(유산·사산 포함)한 건강보험 가입자 또는 피부양자 중 임신·출산
진료비 지원 신청자
  ▸ 핵심내용: • 임산부, 2세 미만 영유아의 진료비 및 약제·치료 재료 구입에 사용할 수 있는
이용권(국민행복카드) 제공
•임 신 1회당 100